In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm

import sys
sys.path.append("../")
from src.synthetic_catalog_creation import synthetic_catalog_creation
from src.synthetic_catalog_creation import focmech_perturbation
from src.synthetic_catalog_creation import stress_inverse

In [ ]:
output_dir = Path("../data/synthetic")
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
### input parameters ###
sigma_model = np.array([[1,0,0], [0,-1,0], [0,0,0]])
M0 = -2
Mc = 2
friction_model = 0.3
focmech_uncertainty = np.array([15, 34])    # degree
N_earthquake = 10**4
N_calc_once = 10**6

stressinverse_dir = "../../Stressinverse_1.1.3"

# relationship between instability and b-value
def model_i_to_b(instability, model_type):
    if model_type == "exp":
        return 1.1 - 0.4 * np.exp((instability - 1) / 0.4)
    if model_type == "linear":
        return np.where(
            instability >= 0.7,
            4 / 3 *(1 - instability) + 0.7,
            1.1
        )

In [ ]:
columns = ["magnitude", "strike", "dip", "rake", "Snn", "Sne", "Snd", "See", "Sed", "Sdd", "R"]

catalog_no_error, catalog_moderate_error, catalog_large_error = [
    pd.DataFrame(columns=columns)
    for _ in range(3)
]

In [ ]:
### catalog creation (exp) ###
## no error
strike_orig, dip_orig, rake_orig, magnitude = synthetic_catalog_creation(
                                                sigma_model,
                                                model_i_to_b,
                                                "exp",
                                                M0,
                                                Mc,
                                                friction_model,
                                                N_earthquake,
                                                N_calc_once,
                                                seed=0
                                                )
catalog_no_error["magnitude"] = magnitude
catalog_no_error["strike"] = strike_orig
catalog_no_error["dip"] = dip_orig
catalog_no_error["rake"] = rake_orig
catalog_no_error["Snn"] = sigma_model[0,0]
catalog_no_error["Sne"] = sigma_model[0,1]
catalog_no_error["Snd"] = sigma_model[0,2]
catalog_no_error["See"] = sigma_model[1,1]
catalog_no_error["Sed"] = sigma_model[1,2]
catalog_no_error["Sdd"] = sigma_model[2,2]
catalog_no_error["R"] = 0.5

catalog_no_error.to_csv(output_dir/"catalog_no_error_exp.csv", index=False)

In [ ]:
# perturbation of focal mechanisms and stress fields
## moderate error
strike_error1, dip_error1, rake_error1 = focmech_perturbation(
                                            strike_orig,
                                            dip_orig,
                                            rake_orig,
                                            focmech_uncertainty[0],
                                            seed=20
                                            )
catalog_moderate_error["magnitude"] = magnitude
catalog_moderate_error["strike"] = strike_error1
catalog_moderate_error["dip"] = dip_error1
catalog_moderate_error["rake"] = rake_error1

stress_columns = ["Snn", "Sne", "Snd", "See", "Sed", "Sdd"]

for i in tqdm(range(100), desc="stres inversion (1)"):
    start = 100 * i
    end = 100 * (i+1)

    sigma_inv1, R_inv1 = stress_inverse(
                            strike_error1[start:end],
                            dip_error1[start:end],
                            rake_error1[start:end],
                            stressinverse_dir
                            )

    stress_values = [
        sigma_inv1[0, 0], sigma_inv1[0, 1], sigma_inv1[0, 2],
        sigma_inv1[1, 1], sigma_inv1[1, 2], sigma_inv1[2, 2]
    ]

    catalog_moderate_error.loc[start:end-1, stress_columns] = stress_values
    catalog_moderate_error.loc[start:end-1, "R"] = R_inv1

catalog_moderate_error.to_csv(output_dir/"catalog_moderate_error_exp.csv", index=False)

In [ ]:
# perturbation of focal mechanisms and stress fields
## large error
strike_error2, dip_error2, rake_error2 = focmech_perturbation(
                                            strike_orig,
                                            dip_orig,
                                            rake_orig,
                                            focmech_uncertainty[1],
                                            seed=30
                                            )
catalog_large_error["magnitude"] = magnitude
catalog_large_error["strike"] = strike_error2
catalog_large_error["dip"] = dip_error2
catalog_large_error["rake"] = rake_error2

stress_columns = ["Snn", "Sne", "Snd", "See", "Sed", "Sdd"]

for i in tqdm(range(100), desc="stress inversion (2)"):
    start = 100 * i
    end = 100 * (i+1)

    sigma_inv2, R_inv2 = stress_inverse(
                            strike_error2[start:end],
                            dip_error2[start:end],
                            rake_error2[start:end],
                            stressinverse_dir
                            )

    stress_values = [
        sigma_inv2[0, 0], sigma_inv2[0, 1], sigma_inv2[0, 2],
        sigma_inv2[1, 1], sigma_inv2[1, 2], sigma_inv2[2, 2]
    ]

    catalog_large_error.loc[start:end-1, stress_columns] = stress_values
    catalog_large_error.loc[start:end-1, "R"] = R_inv2

catalog_large_error.to_csv(output_dir/"catalog_large_error_exp.csv", index=False)

In [ ]:
### catalog creation (linear) ###
## no error
strike_orig, dip_orig, rake_orig, magnitude = synthetic_catalog_creation(
                                                sigma_model,
                                                model_i_to_b,
                                                "linear",
                                                M0,
                                                Mc,
                                                friction_model,
                                                N_earthquake,
                                                N_calc_once,
                                                seed=0
                                                )
catalog_no_error["magnitude"] = magnitude
catalog_no_error["strike"] = strike_orig
catalog_no_error["dip"] = dip_orig
catalog_no_error["rake"] = rake_orig
catalog_no_error["Snn"] = sigma_model[0,0]
catalog_no_error["Sne"] = sigma_model[0,1]
catalog_no_error["Snd"] = sigma_model[0,2]
catalog_no_error["See"] = sigma_model[1,1]
catalog_no_error["Sed"] = sigma_model[1,2]
catalog_no_error["Sdd"] = sigma_model[2,2]
catalog_no_error["R"] = 0.5

catalog_no_error.to_csv(output_dir/"catalog_no_error_linear.csv", index=False)

In [ ]:
# perturbation of focal mechanisms and stress fields
## moderate error
strike_error1, dip_error1, rake_error1 = focmech_perturbation(
                                            strike_orig,
                                            dip_orig,
                                            rake_orig,
                                            focmech_uncertainty[0],
                                            seed=40
                                            )
catalog_moderate_error["magnitude"] = magnitude
catalog_moderate_error["strike"] = strike_error1
catalog_moderate_error["dip"] = dip_error1
catalog_moderate_error["rake"] = rake_error1

stress_columns = ["Snn", "Sne", "Snd", "See", "Sed", "Sdd"]

for i in tqdm(range(100), desc="stres inversion (1)"):
    start = 100 * i
    end = 100 * (i+1)

    sigma_inv1, R_inv1 = stress_inverse(
                            strike_error1[start:end],
                            dip_error1[start:end],
                            rake_error1[start:end],
                            stressinverse_dir
                            )

    stress_values = [
        sigma_inv1[0, 0], sigma_inv1[0, 1], sigma_inv1[0, 2],
        sigma_inv1[1, 1], sigma_inv1[1, 2], sigma_inv1[2, 2]
    ]

    catalog_moderate_error.loc[start:end-1, stress_columns] = stress_values
    catalog_moderate_error.loc[start:end-1, "R"] = R_inv1

catalog_moderate_error.to_csv(output_dir/"catalog_moderate_error_linear.csv", index=False)

In [ ]:
# (perturbation of focal mechanisms and stress fields)
## large error
strike_error2, dip_error2, rake_error2 = focmech_perturbation(
                                            strike_orig,
                                            dip_orig,
                                            rake_orig,
                                            focmech_uncertainty[1],
                                            seed=50
                                            )
catalog_large_error["magnitude"] = magnitude
catalog_large_error["strike"] = strike_error2
catalog_large_error["dip"] = dip_error2
catalog_large_error["rake"] = rake_error2

stress_columns = ["Snn", "Sne", "Snd", "See", "Sed", "Sdd"]

for i in tqdm(range(100), desc="stress inversion (2)"):
    start = 100 * i
    end = 100 * (i+1)

    sigma_inv2, R_inv2 = stress_inverse(
                            strike_error2[start:end],
                            dip_error2[start:end],
                            rake_error2[start:end],
                            stressinverse_dir
                            )

    stress_values = [
        sigma_inv2[0, 0], sigma_inv2[0, 1], sigma_inv2[0, 2],
        sigma_inv2[1, 1], sigma_inv2[1, 2], sigma_inv2[2, 2]
    ]

    catalog_large_error.loc[start:end-1, stress_columns] = stress_values
    catalog_large_error.loc[start:end-1, "R"] = R_inv2

catalog_large_error.to_csv(output_dir/"catalog_large_error_linear.csv", index=False)